# 01 · The raw model

Research notebook: this is evidence, not a usage guide (see `notebooks/` for that).
It does **not** use the engine. It calls `sentence-transformers` and `transformers`
directly, to record the behaviour the engine is built around. Every finding here is listed in
[`docs/model-facts.md`](../docs/model-facts.md), and each "silent" behaviour becomes a rule in the engine.

Rerun it after upgrading `transformers` or `sentence-transformers`, or after changing the pinned revision.

In [1]:
import gc
import os
import warnings
from pathlib import Path

import numpy as np
import sentence_transformers
import torch
import transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor

import embeddinggemma_engine as eg

warnings.filterwarnings("ignore")
transformers.utils.logging.disable_progress_bar()
if Path.cwd().name == "research":
    os.chdir("..")  # run from the repo root, so paths stay short
ROOT = Path(".")
A = ROOT / "assets"
MODEL, REV = eg.MODEL_ID, eg.PINNED_REVISION
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))


def load(dtype=torch.bfloat16, **config_kwargs):
    return SentenceTransformer(MODEL, revision=REV, local_files_only=True, device="cuda",
                               model_kwargs={"dtype": dtype}, config_kwargs=config_kwargs)

sentence-transformers 6.1.0 | transformers 5.19.0 | torch 2.14.0+cu130 | NVIDIA GeForce RTX 4070 Ti SUPER


## 1. Size and VRAM per set of towers
`config_kwargs` can drop the vision and/or audio tower at load time.

In [2]:
for name, cfg in [("text", {"vision_config": None, "audio_config": None}),
                  ("text+vision", {"audio_config": None}),
                  ("text+audio", {"vision_config": None}),
                  ("full", {})]:
    m = load(**cfg)
    params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{name:12s} {params:5.0f}M params  {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    del m; gc.collect(); torch.cuda.empty_cache()

text           271M params  0.54 GB


text+vision    439M params  0.89 GB


text+audio     577M params  1.15 GB


full           744M params  1.50 GB


In [3]:
model = load()
processor = model[0].processor
print(model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'image': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'audio': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'video': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'message': {'method': 'forward', 'method_output_name': 'last_hidden_state', 'format': 'structured'}}, 'module_output_name': 'token_embeddings', 'architecture': 'EmbeddingGemma2Model'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


## 2. Prefixes
A manually built prefix is identical to `prompt_name=`. Leaving the prefix out gives a noticeably different vector.

In [4]:
q = "how do I make naan without a tandoor"
by_name = model.encode(q, prompt_name="SearchQuery")
manual = model.encode("task: search result | query: " + q)
bare = model.encode(q)
print("prompt_name vs manual, max abs diff:", np.abs(by_name - manual).max())
print("with vs without prefix, cosine:     ", round(float(by_name @ bare), 4))
print("Document prompt:", repr(model.prompts["Document"]), "<- title is always 'none'")

prompt_name vs manual, max abs diff: 0.0
with vs without prefix, cosine:      0.9106
Document prompt: 'title: none | text: ' <- title is always 'none'


## 3. `truncate_dim` does not re-normalize

In [5]:
for dim in (768, 512, 256, 128):
    v = model.encode(q, truncate_dim=dim)
    print(f"dim {dim}: norm {np.linalg.norm(v):.4f}")

dim 768: norm 1.0013
dim 512: norm 0.8852
dim 256: norm 0.7182


dim 128: norm 0.6089


## 4. No length limit, and bare strings are guessed as media

In [6]:
print("max_seq_length:", model.max_seq_length)
long_text = "word " * 12000
print("12k-word text embeds without complaint:", model.encode(long_text).shape)
from sentence_transformers.base.modality import infer_modality  # noqa: E402

for s in [str(A / "images/kitchen.jpg"), "https://example.com/cat.jpg", "photo.jpg",
          "see https://example.com/cat.jpg"]:
    print(f"{infer_modality(s):6s} <- {s[-40:]!r}")

max_seq_length: 1000000000000000019884624838656


12k-word text embeds without complaint: (768,)
image  <- 'assets/images/kitchen.jpg'
image  <- 'https://example.com/cat.jpg'
text   <- 'photo.jpg'
text   <- 'see https://example.com/cat.jpg'


## 5. fp16: fine on short text, NaN on long text, with no error

In [7]:
del model; gc.collect(); torch.cuda.empty_cache()
m16 = load(torch.float16)
print("short fp16 NaN:", np.isnan(m16.encode(q)).any())
print("long  fp16 NaN:", np.isnan(m16.encode(long_text)).any())
del m16; gc.collect(); torch.cuda.empty_cache()
model = load()

short fp16 NaN: False


long  fp16 NaN: True


## 6. Token costs and silent media limits (processor only)

In [8]:
def tokens(content, **kw):
    out = processor.apply_chat_template([{"role": "user", "content": content}], tokenize=True,
                                        return_dict=True, return_tensors="pt", **kw)
    ids = out["input_ids"][0].tolist()
    return {"total": len(ids), "image": ids.count(processor.image_token_id),
            "video": ids.count(processor.video_token_id), "audio": ids.count(processor.audio_token_id)}


img, vid10, vid35 = A / "images/wildlife.jpg", A / "videos/05_juggle.mp4", A / "videos/30s_vid.mp4"
print("image, default         ", tokens([{"type": "image", "url": str(img)}]))
print("image, budget 70       ", tokens([{"type": "image", "url": str(img)}], processor_kwargs={"images_kwargs": {"max_soft_tokens": 70}}))
print("video 10.3 s           ", tokens([{"type": "video", "url": str(vid10)}]))
print("video 34.8 s (capped!) ", tokens([{"type": "video", "url": str(vid35)}]), "<- 32 frames x 120, 2 frames dropped")
print("video 34.8 s, max 64   ", tokens([{"type": "video", "url": str(vid35)}], processor_kwargs={"videos_kwargs": {"max_frames": 64}}))

image, default          {'total': 280, 'image': 276, 'video': 0, 'audio': 0}
image, budget 70        {'total': 70, 'image': 66, 'video': 0, 'audio': 0}
video 10.3 s            {'total': 1222, 'image': 0, 'video': 1200, 'audio': 0}


video 34.8 s (capped!)  {'total': 3906, 'image': 0, 'video': 3840, 'audio': 0} <- 32 frames x 120, 2 frames dropped


video 34.8 s, max 64    {'total': 4150, 'image': 0, 'video': 4080, 'audio': 0}


In [9]:
silence = lambda s: np.zeros(16_000 * s, np.float32)  # noqa: E731
print("audio 10 s  ", tokens([{"type": "audio", "audio": silence(10)}]))
print("audio 120 s ", tokens([{"type": "audio", "audio": silence(120)}]), "<- cut at 30 s")
print("audio 120 s, truncation=False", tokens([{"type": "audio", "audio": silence(120)}],
                                              processor_kwargs={"audio_kwargs": {"truncation": False}}))
print("10 s array at 44.1 kHz, sampling_rate passed:",
      tokens([{"type": "audio", "audio": np.zeros(44_100 * 10, np.float32)}],
             processor_kwargs={"audio_kwargs": {"sampling_rate": 44_100}}), "<- read as 27.6 s")
try:
    tokens([{"type": "audio", "audio": np.zeros((2, 44_100), np.float32)}])
except Exception as ex:
    print("stereo array:", type(ex).__name__, str(ex)[:80])

audio 10 s   {'total': 254, 'image': 0, 'video': 0, 'audio': 250}
audio 120 s  {'total': 754, 'image': 0, 'video': 0, 'audio': 750} <- cut at 30 s
audio 120 s, truncation=False {'total': 3004, 'image': 0, 'video': 0, 'audio': 3000}
10 s array at 44.1 kHz, sampling_rate passed: {'total': 693, 'image': 0, 'video': 0, 'audio': 689} <- read as 27.6 s
stereo array: ValueError operands could not be broadcast together with remapped shapes [original->remappe


In [10]:
print("video soundtrack off (default):", tokens([{"type": "video", "url": str(A / "videos/04_coffee.mp4")}]))
print("video soundtrack on:           ", tokens([{"type": "video", "url": str(A / "videos/04_coffee.mp4")}], load_audio_from_video=True))
try:
    tokens([{"type": "video", "url": str(vid10)}], load_audio_from_video=True)
except Exception as ex:
    print("soundtrack on a silent video:", type(ex).__name__, str(ex).splitlines()[0])

video soundtrack off (default): {'total': 1100, 'image': 0, 'video': 1080, 'audio': 0}
video soundtrack on:            {'total': 1341, 'image': 0, 'video': 1080, 'audio': 239}
soundtrack on a silent video: ValueError The best audio stream is unknown and there is no specified stream. 


## 7. Should media documents get a prefix?
Text queries (with the search prefix) are compared against media documents in two forms.
The margin is the gap between the best and second-best score, averaged over queries.

In [11]:
import json  # noqa: E402

queries = [json.loads(line) for line in open(A / "text/queries.jsonl")]
captions = [json.loads(line) for line in open(A / "text/captions.jsonl")]
for kind in ("image", "video"):
    paths = [c["path"] for c in captions if f"/{kind}" in c["path"]]
    qq = [x for x in queries if x["target"] == kind]
    Q = model.encode(["task: search result | query: " + x["query"] for x in qq], normalize_embeddings=True)
    for label, pre in [("no prefix", None), ("'title: none | text: '", "title: none | text: ")]:
        docs = [[{"role": "user", "content": ([{"type": "text", "text": pre}] if pre else [])
                  + [{"type": kind, "url": str(ROOT / p)}]}] for p in paths]
        D = model.encode(docs, batch_size=4, normalize_embeddings=True)
        S = Q @ D.T
        top1 = sum(paths[int(np.argmax(s))] in x["expected"] for s, x in zip(S, qq))
        margin = np.mean(np.sort(S, 1)[:, -1] - np.sort(S, 1)[:, -2])
        print(f"{kind:5s} {label:24s} top1 {top1}/{len(qq)}  margin {margin:.3f}")

image no prefix                top1 5/5  margin 0.142


image 'title: none | text: '   top1 5/5  margin 0.122


video no prefix                top1 4/4  margin 0.125


video 'title: none | text: '   top1 4/4  margin 0.121


**Conclusions** (copied into `docs/model-facts.md`):
- Media-only items work best without a prefix. The engine adds a prefix only when an item has text or a title.
- The engine enforces everything the libraries leave silent: the context limit, re-normalization,
  the video frame cap, audio over 30 s, array sampling rates, and refusing fp16.